In [5]:
# 파일: autogluon_run.py (예시)

import pandas as pd
from autogluon.tabular import TabularPredictor

# 1) 데이터 로드
train_path = "../data/train.tsv"
test_path = "../data/test.tsv"

# TSV 이므로 sep="\t" 사용
train_df = pd.read_csv(train_path, sep="\t")
test_df = pd.read_csv(test_path, sep="\t")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# ─────────────────────────────────────────
# ID 컬럼 제거 (id 컬럼명이 다를 경우 수정)
if "train_id" in train_df.columns:
    train_df = train_df.drop(columns=["train_id"])
if "test_id" in test_df.columns:
    test_df = test_df.drop(columns=["test_id"])
# ─────────────────────────────────────────

# 2) 타깃 / ID 컬럼 이름 설정
TARGET_COL = "price"   # 실제 타깃 컬럼명으로 바꿔 주세요
ID_COL = None           # 제출용 ID 컬럼명으로 바꿔 주세요 (없으면 None)

# 3) AutoGluon 학습
# presets / time_limit 등은 상황에 맞게 조절
predictor = TabularPredictor(
    label=TARGET_COL,
    problem_type=None,          # 회귀/분류 자동 추론, 명시하고 싶으면 "regression"/"multiclass"/"binary"
    path="autogluon_models"     # 모델이 저장될 폴더
).fit(
    train_data=train_df,
    presets="medium_quality_faster_train",  # 빠른 실험용
    time_limit=3600,                        # 최대 1시간 (초 단위), 필요에 따라 조정
)

# 4) 리더보드 확인 (optional)
leaderboard_df = predictor.leaderboard(silent=True)
print(leaderboard_df.head())

# 5) 테스트 데이터 예측
test_preds = predictor.predict(test_df)

# 6) 제출 파일 생성
if ID_COL in test_df.columns:
    submission = pd.DataFrame({
        ID_COL: test_df[ID_COL],
        TARGET_COL: test_preds
    })
else:
    # ID 컬럼이 없다면 단순히 index 기반으로 생성
    submission = pd.DataFrame({
        "id": range(len(test_preds)),
        TARGET_COL: test_preds
    })

submission_path = "submission_autogluon.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)


Preset alias specified: 'medium_quality_faster_train' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.10.19
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.19045
CPU Count:          12
Memory Avail:       21.49 GB / 31.91 GB (67.3%)
Disk Space Avail:   309.83 GB / 465.15 GB (66.6%)
Presets specified: ['medium_quality_faster_train']
Using hyperparameters preset: hyperparameters='default'


Train shape: (1482535, 8)
Test shape: (693359, 7)


Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "c:\big20\git\big20-ML-project2-team3\MercariPriceSuggestion\src\autogluon_models"
Train Data Rows:    1482535
Train Data Columns: 6
Label Column:       price
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and label-values can't be converted to int).
	Label info (max, min, mean, stddev): (2009.0, 0.0, 26.73752, 38.58607)
	If 'regression' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    22531.81 MB
	Train Data (Original)  Memory Usage: 656.23 MB (2.9% of available memory)
	Inferring data type of each feature based on column va

[1000]	valid_set's rmse: 27.5082
[2000]	valid_set's rmse: 27.0844
[3000]	valid_set's rmse: 26.9491
[4000]	valid_set's rmse: 26.809
[5000]	valid_set's rmse: 26.7203
[6000]	valid_set's rmse: 26.6738
[7000]	valid_set's rmse: 26.6122
[8000]	valid_set's rmse: 26.5828
[9000]	valid_set's rmse: 26.5566
[10000]	valid_set's rmse: 26.5444


	-26.534	 = Validation score   (-root_mean_squared_error)
	319.71s	 = Training   runtime
	3.82s	 = Validation runtime
Fitting model: LightGBM ... Training model for up to 2686.19s of the 2686.19s of remaining time.
	Fitting with cpus=6, gpus=0, mem=7.6/18.6 GB


[1000]	valid_set's rmse: 26.9113
[2000]	valid_set's rmse: 26.635
[3000]	valid_set's rmse: 26.5196
[4000]	valid_set's rmse: 26.4612
[5000]	valid_set's rmse: 26.3813
[6000]	valid_set's rmse: 26.3702
[7000]	valid_set's rmse: 26.364
[8000]	valid_set's rmse: 26.3418
[9000]	valid_set's rmse: 26.3191
[10000]	valid_set's rmse: 26.3125


	-26.3111	 = Validation score   (-root_mean_squared_error)
	313.27s	 = Training   runtime
	3.29s	 = Validation runtime
Fitting model: RandomForestMSE ... Training model for up to 2368.56s of the 2368.56s of remaining time.
	Fitting with cpus=12, gpus=0, mem=0.9/18.6 GB
	-29.4797	 = Validation score   (-root_mean_squared_error)
	1824.55s	 = Training   runtime
	0.14s	 = Validation runtime
Fitting model: CatBoost ... Training model for up to 543.79s of the 543.79s of remaining time.
	Fitting with cpus=6, gpus=0, mem=8.4/21.6 GB
		`import catboost` failed. A quick tip is to install via `pip install autogluon.tabular[catboost]==1.4.0`.
Fitting model: ExtraTreesMSE ... Training model for up to 538.95s of the 538.95s of remaining time.
	Fitting with cpus=12, gpus=0, mem=0.9/21.4 GB
	Time limit exceeded... Skipping ExtraTreesMSE.
Fitting model: NeuralNetFastAI ... Training model for up to 265.46s of the 265.45s of remaining time.
	To avoid this warning, specify the model hyperparameter "ag.max

                 model  score_val              eval_metric  pred_time_val  \
0  WeightedEnsemble_L2 -26.092841  root_mean_squared_error       7.434983   
1             LightGBM -26.311065  root_mean_squared_error       3.294976   
2           LightGBMXT -26.533985  root_mean_squared_error       3.824001   
3      NeuralNetFastAI -28.891443  root_mean_squared_error       0.174000   
4      RandomForestMSE -29.479705  root_mean_squared_error       0.141007   

      fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  \
0  2712.942396                0.001000           0.027010            2   
1   313.266201                3.294976         313.266201            1   
2   319.710993                3.824001         319.710993            1   
3   255.391567                0.174000         255.391567            1   
4  1824.546626                0.141007        1824.546626            1   

   can_infer  fit_order  
0       True          5  
1       True          2  
2       True  